## Cosmic Web - CppJIT - CUDA Kernel

[CppJIT](https://github.com/compiler-research/cppjit) lets us compile C++ and CUDA source code inside the Python session at runtime. `cppjit.gbl.<name>` is the bound compiled function or kernel, and CuPy's arrays and streams can then be passed into a launch. This is the *kernel* and *interoperability* model. In this notebook we call CUB's histogram from C++ first, then write the kernel ourselves and measure the costs.

### Table of Contents

1. [Setup](#1-setup)
2. [CUB from C++](#2-cub-from-c)
3. [Kernel](#3-kernel)
4. [Launch](#4-launch)
5. [Trace and bytes](#5-trace-and-bytes)
6. [Limitations and recap](#6-limitations-and-recap)

---

### 1. Setup

`import cppjit` starts a CUDA-enabled C++ interpreter ([clang-repl](https://clang.llvm.org/docs/ClangRepl.html)) in this process.

---

**Quick Docs**

- `cppjit.cppdef(src)`: compile C++ or CUDA source into the session; its functions and kernels appear as `cppjit.gbl.<name>`.

In [ ]:
import os
import time

import numpy as np
import cupy as cp

import cosmic_core as cw            # sets the interpreter's optimisation level before cppjit starts
import cppjit

assert cppjit.CUDA_ENABLED, 'this notebook needs the CUDA-enabled tutorial image'

N, K = cw.setup('cppjit_cub', f"interpreter arguments {os.environ['CPPINTEROP_EXTRA_INTERPRETER_ARGS']}")

### 2. CUB from C++

`cosmic_cub_histogram.cu` wraps `DeviceHistogram` from [CUB](https://github.com/NVIDIA/cccl/tree/main/cub) in one host function that takes the device pointers and the stream as integers. Let's compile it into the session, count the eight cell arrays of the CuPy reference with it in place of `cp.bincount`, and make sure it works.

---

**Quick Docs**

- `cppjit.gbl.cub_histogram(samples_ptr, hist_ptr, n, k_bins, stream)`: CUB's `HistogramEven` over `n` int32 samples into `k_bins` counters.
- `a.data.ptr`, `stream.ptr`: a CuPy array's device address and a stream's handle, as integers.

In [ ]:
t0 = time.perf_counter()
cppjit.cppdef(cw.cub_histogram_source())
cppdef_cub_s = time.perf_counter() - t0
print(f'cppdef of the CUB wrapper: {cppdef_cub_s * 1e3:.0f} ms')

SIGNS = (+1, -1, -1, -1, +1, +1, +1, -1)
CUB_HISTS = [cp.zeros(K, dtype=cp.int32) for _ in SIGNS]


def ecc_hist_cub(q):
    '''Signed histogram of births (host int64, length K): the CuPy stencil, CUB's histogram from C++ per cell array.'''
    v, (ex, ey, ez), (fxy, fxz, fyz), c = cw.ecc_cells_cupy(q)
    stream = cp.cuda.get_current_stream()
    for a, h in zip((v, ex, ey, ez, fxy, fxz, fyz, c), CUB_HISTS):
        cppjit.gbl.cub_histogram(a.data.ptr, h.data.ptr, a.size, K, stream.ptr)
    # The signed sum of the eight counts, on the host, is the token histogram.
    return sum(sign * cp.asnumpy(h).astype(np.int64) for sign, h in zip(SIGNS, CUB_HISTS))


def run_cub(n):
    return ecc_hist_cub(cw.field(n))


_ = cw.measure('cppjit_cub', run_cub, N, compile_s=cppdef_cub_s, cold_note='cppdef and first call')

### 3. Kernel

The kernel in `cosmic_ecc.cu` gives thread $t$ the voxel $(i, j, k)$ with $k$ varying fastest, so the loads of a warp are coalesced.

- Each thread loads its periodic $2\times2\times2$ window and takes the eight births with `min`.
- It adds $\pm 1$ at each birth bin into a `__shared__` histogram, the on-chip memory shared by the threads of a block.
- After `__syncthreads()`, a block-wide barrier, the block adds its non-zero counters into the histogram in global memory with `atomicAdd`.

Why shared memory? It privatises the histogram, one copy per thread block, as the slabs did in the JAX notebook. Without it every voxel would issue eight atomic adds into global memory. With it a thread block issues at most $K$, one per non-zero counter.

The TODO writes the body between the two markers. The zeroing, the periodic indices, the barrier and the flush of the block's counters into the global histogram are written.

---

**Quick Docs**

- `__global__ void kernel(...)`: a [CUDA C++](https://docs.nvidia.com/cuda/cuda-programming-guide/) kernel, launched on a grid of thread blocks; `blockIdx`, `blockDim` and `threadIdx` give each thread its index.
- `atomicAdd(int*, int)`, `atomicSub(int*, int)`: thread-safe read-modify-write add or subtract at an address in shared or global memory.
- `cw.ecc_cuda_source(body)`: `cosmic_ecc.cu` with `body` spliced between its `BEGIN BODY` and `END BODY` markers.

In [ ]:
KERNEL_BODY = r'''
        // TODO: the eight births and the eight shared-memory atomics. The corners are q[i * nn + j * n + k] and its
        // neighbours at i1, j1, k1 (v000 ... v111); a cell is born at the minimum over its corners.
        // atomicAdd(&sh[b], 1) at v000, fxy, fxz, fyz; atomicSub(&sh[b], 1) at ex, ey, ez, c.
'''

t0 = time.perf_counter()
cppjit.cppdef(cw.ecc_cuda_source(body=KERNEL_BODY))
cppdef_s = time.perf_counter() - t0
print(f'cppdef of the kernel: {cppdef_s * 1e3:.0f} ms')

### 4. Launch

The grid gives one thread per voxel. The shared histogram is static, so the dynamic shared-memory slot is zero. The stream slot takes CuPy's current stream, and a CuPy array pointer is passed to the kernel: cppjit reads the array's device address through its array interface.

The TODO writes the launch. The histogram buffer, the grid and the runner are written.

---

**Quick Docs**

- `cppjit.gbl.<kernel>[grid, block, shared_bytes, stream](*args)`: the launch; the bracket is the *execution configuration*: `grid` and `block` are ints or 3-tuples, `shared_bytes` the dynamic shared memory, `stream` an int handle or any `__cuda_stream__` object.

In [ ]:
BLOCK = 256


def ecc_hist_cppjit(q):
    '''Signed histogram of births (int32, length K) on the device: one launch on the current stream.'''
    n = q.shape[0]
    hist = cp.zeros(K, dtype=cp.int32)
    stream = cp.cuda.get_current_stream()
    grid = (n ** 3 + BLOCK - 1) // BLOCK
    # TODO: launch cppjit.gbl.ecc_hist_kernel on grid blocks of BLOCK threads, no dynamic shared memory, on stream, with q, hist and n.
    ...
    return hist


def run_cppjit(n):
    hist = ecc_hist_cppjit(cw.field(n))
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist).astype(np.int64)

Now let's make sure it works. The cold figure is the `cppdef` above plus the first launch, where cppjit compiles a launcher for the `<<<>>>` call and the driver JIT-compiles the PTX for the device.

In [ ]:
warm = cw.measure('cppjit_cuda', run_cppjit, N, compile_s=cppdef_s, cold_note='cppdef and first launch')

### 5. Trace and bytes

One call should launch one kernel, one memset and one copy back. The bytes model counts one read of the field, four bytes per voxel: the other seven loads of each thread should hit in cache, and the flush should be served by L2.

In [ ]:
cw.trace_call('cppjit_cuda', run_cppjit, N, [ecc_hist_cppjit], header=f'''
import cppjit
BLOCK = {BLOCK}
cppjit.cppdef(cw.ecc_cuda_source(body={KERNEL_BODY!r}))
''')
cw.roof_report('cppjit_cuda', warm)
cw.save_sweep('cppjit_cuda', run_cppjit)

**EXTRA CREDIT: profile**

The warm time sits above the roof for one read of the field. [Nsight Compute](https://docs.nvidia.com/nsight-compute/) replays one launch and reads the GPU's hardware counters. Its report tells us how busy the *SMs* (streaming multiprocessors) and the memory system were, what the warps waited for, and which level of the memory hierarchy served each load.

---

**Quick Docs**

- `cw.ncu_report(stage, n, title, run, functions, args, header)`: `ncu --set full` on one launch of the stage's probe, written from `run` when given; prints the metrics that matter here and the top warp stall reasons.

In [ ]:
# EXTRA CREDIT: the kernel of section 3 on the profiler.
cw.ncu_report('cppjit_cuda', N, f'one launch at N = {N}^3:')

What does the report say?

- SM throughput is higher than memory throughput: instruction issue limits the kernel.
- The top stall reason, `wait`, is a fixed-latency execution dependency on the previous instruction: here the integer divisions that turn the thread index into $(i, j, k)$.
- Each thread loads eight corners, so the kernel requests the field several times over. most requests hit in L1 or L2, and the field is read from DRAM about once: the traffic the bytes model counts.
- The per-voxel atomics stay in shared memory. The flush's atomic adds are performed as reductions in L2, so DRAM traffic stays at one read of the field.

What if the grid gave each thread its $i$ and $j$ directly? `ecc_hist_rows_kernel` in `cosmic_ecc_variants.cu` runs the same body on a 3D grid: the block indices `blockIdx.y` and `blockIdx.z` give $j$ and $i$ directly, and each thread walks several voxels along $k$. The TODO is its launch on a grid of `(1, n, n)`, one thread block per row. The block size sets how many voxels a thread walks: sweep it, then profile one.

In [ ]:
# EXTRA CREDIT: the rows kernel.
cppjit.cppdef(cw.ecc_cuda_variants_source())


def ecc_hist_rows(q, block):
    '''Signed histogram of births (int32, length K) on the device, through ecc_hist_rows_kernel.'''
    n = q.shape[0]
    hist = cp.zeros(K, dtype=cp.int32)
    stream = cp.cuda.get_current_stream()
    # TODO: launch cppjit.gbl.ecc_hist_rows_kernel on the grid (1, n, n) with block threads, no dynamic shared memory, on stream, with q, hist and n.
    ...
    return hist


def run_rows(n, block):
    hist = ecc_hist_rows(cw.field(n), block)
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist).astype(np.int64)

In [ ]:
print(f"{'block':>6} {'voxels/thread':>13} {'warm ms':>9} {'GB/s field':>11}")
for block in (512, 256, 128, 64):
    r = cw.measure('rows', run_rows, N, block, record=False, quiet=True)
    print(f'{block:>6} {(N + block - 1) // block:>13} {r["median_s"] * 1e3:>9.3f} {cw.field(N).nbytes / r["median_s"] / 1e9:>11.0f}')

ROWS_BLOCK = 128
cw.ncu_report('rows', N, f'\nthe rows kernel, {(N + ROWS_BLOCK - 1) // ROWS_BLOCK} voxels per thread, one launch at N = {N}^3:',
              run=run_rows, functions=[ecc_hist_rows], args=(ROWS_BLOCK,), header='''
import cppjit
cppjit.cppdef(cw.ecc_cuda_variants_source())
''')

- On the 3D grid the divisions are gone.
- The memory throughput is now the higher of the two, and the top stall, `long_scoreboard`, is a scoreboard dependency on a global-memory load: memory limits the kernel.
- With more voxels per thread the field-read rate climbs, then flattens below the copy roof.

Takeaways:

- The shared-memory histogram turns $8N^3$ global atomics into at most $K$ per thread block.
- CuPy's arrays and stream went into CUB's function and into the kernel with no copy or binding.

### 6. Limitations and recap

A name defined in one `cppdef` stays defined for the session, so a second version of a kernel has to be defined under a new name. A field larger than the device's memory is a future direction: slab decomposition with a halo plane.

Recap:

- CUB's histogram and the kernel ran from source compiled in the session, on CuPy's buffers.

Next: `95__cosmic_web__synthesis.ipynb` reads the stage files and draws the comparison.